# Cascad — Hugging Face attribution baseline

This notebook runs one frozen local model at a time on Kaggle or Google Colab. Select a GPU runtime before starting. Run `qwen3-4b` first; use a fresh session for `mistral-7b` if disk space is limited.

In [ ]:
import importlib
import os
import pathlib
import subprocess
import sys

REPO_URL = os.environ.get("CASCAD_REPO_URL", "https://github.com/elom354/cascad.git")
MODEL_ALIAS = os.environ.get("CASCAD_HF_MODEL", "qwen3-4b")
assert MODEL_ALIAS in {"qwen3-4b", "mistral-7b"}

base = pathlib.Path("/kaggle/working" if pathlib.Path("/kaggle/working").exists() else "/content")
repo = base / "Cascad"
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
print({"repository": str(repo), "model": MODEL_ALIAS})

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[huggingface]"], check=True)
src = str(repo / "src")
os.environ["PYTHONPATH"] = src + os.pathsep + os.environ.get("PYTHONPATH", "")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.import_module("cascad")
torch = importlib.import_module("torch")
assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook settings"
print({"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)})

In [ ]:
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception as exc:
    print("Impossible de charger HF_TOKEN :", exc)

print("HF token configured:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
output = base / f"cascad-huggingface-{MODEL_ALIAS}-sdpa-offloaded"
command = [
    sys.executable,
    "scripts/run_huggingface_attribution.py",
    "--models", MODEL_ALIAS,
    "--quantization", "4bit",
    "--attention-backend", "sdpa",
    "--cache-implementation", "offloaded",
    "--out", str(output),
]
print(" ".join(command))
process = subprocess.Popen(
    command,
    cwd=repo,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tail = []
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
    tail.append(line)
    tail = tail[-80:]
return_code = process.wait()
if return_code:
    raise RuntimeError(
        f"Hugging Face runner failed with exit code {return_code}.\n"
        + "".join(tail)
    )

In [ ]:
import json
import shutil

summary_path = output / "summary.json"
assert summary_path.is_file(), "The runner did not produce summary.json"
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
assert summary["study_complete"], "Not all frozen instances completed successfully"
archive = shutil.make_archive(str(output), "zip", output)
print("Download this archive before closing the session:", archive)